In [17]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

In [18]:
df_a = pd.read_csv('Module 5/student-por.csv', sep = ";")
print("Initial shape:", df_a.shape)
df_a.head()

Initial shape: (649, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [19]:
n_duplicates = df_a.duplicated().sum()
print(f"Found {n_duplicates} duplicate rows.")

Found 0 duplicate rows.


In [20]:
print("\nMissing values per column:")
print(df_a.isnull().sum())


Missing values per column:
school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
G1            0
G2            0
G3            0
dtype: int64


In [21]:
df_b = df_a

In [22]:
# Define target
y = df_b['G3']

# Define features
X = df_b.drop(columns=['G1', 'G2', 'G3'])

# Split before encoding
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
nominal_features = ["school", "sex", "address", "famsize", "Pstatus", "Mjob", "Fjob", "reason", "guardian", "schoolsup", "famsup", "paid", "activities", "nursery", "higher", "internet", "romantic"]

ordinal_features = ["Medu", "Fedu", "traveltime", "studytime", "famrel", "freetime", "goout", "Dalc", "Walc", "health"]

numerical_features = ["age", "absences", "failures"]

In [24]:
# One-hot encode nominal features
OHE = OneHotEncoder(drop='first')
X_train_nominal = OHE.fit_transform(X_train_raw[nominal_features])
X_test_nominal = OHE.transform(X_test_raw[nominal_features])

# Convert to DataFrame
X_train_nominal_df = pd.DataFrame(X_train_nominal, columns=OHE.get_feature_names_out(nominal_features), index=X_train_raw.index)
X_test_nominal_df = pd.DataFrame(X_test_nominal, columns=OHE.get_feature_names_out(nominal_features), index=X_test_raw.index)

ValueError: Shape of passed values is (519, 1), indices imply (519, 26)

In [ ]:
# Ordinal encode ordered features
OE = OrdinalEncoder()
X_train_ordinal = OE.fit_transform(X_train_raw[ordinal_features])
X_test_ordinal = OE.transform(X_test_raw[ordinal_features])

# Convert to DataFrame
X_train_ordinal_df = pd.DataFrame(X_train_ordinal, columns=ordinal_features, index=X_train_raw.index)
X_test_ordinal_df = pd.DataFrame(X_test_ordinal, columns=ordinal_features, index=X_test_raw.index)

In [ ]:
# Keep numeric features as-is
X_train_numeric_df = X_train_raw[numerical_features].copy()
X_test_numeric_df = X_test_raw[numerical_features].copy()

In [ ]:
# Concatenate all feature types
X_train_combined = pd.concat([X_train_nominal_df, X_train_ordinal_df, X_train_numeric_df], axis=1)
X_test_combined = pd.concat([X_test_nominal_df, X_test_ordinal_df, X_test_numeric_df], axis=1)

In [ ]:
# Combine features and target
Xy_train_combined = X_train_combined.copy()
Xy_train_combined['G3'] = y_train
corr_matrix = Xy_train_combined.corr()

plt.figure(figsize=(18, 14))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False, fmt=".2f")
plt.title("Correlation Matrix (Including Target G3)")
plt.show()

In [ ]:
def low_corr_report(df: pd.DataFrame, target: str, thresh: float = 0.3):
    # 1. Compute full correlation matrix
    corr_matrix = df.corr()

    # 2. Extract correlations with the target (drop the target itself)
    corr_with_target = corr_matrix[target].drop(target)

    # 3. Filter by absolute threshold
    low_corr = corr_with_target[corr_with_target.abs() < thresh ]

    # 4. Report
    if low_corr.empty:
        print(f"No predictors have |corr| < {thresh:.2f} with '{target}'.")
    else:
        print(f"Predictors with |corr| < {thresh:.2f} to '{target}':\n")
        display(low_corr.to_frame(name=f"corr_with_{target}"))

    return low_corr.index.to_list()

low_corr = low_corr_report(Xy_train_combined, target='G3', thresh=0.3)

In [ ]:
def get_strong_predictor_correlations(df: pd.DataFrame, target: str, thresh: float = 0.7) -> pd.DataFrame:
    corr_matrix = df.corr()
    
    predictors_corr = corr_matrix.drop(index=target, columns=target)

    # Step 3: Unstack, filter, and deduplicate
    strong_pairs = (predictors_corr.abs().unstack().reset_index().rename(columns={0: 'abs_corr'}))

    strong_pairs = strong_pairs[strong_pairs['level_0'] != strong_pairs['level_1']]

    strong_pairs = strong_pairs[strong_pairs['abs_corr'] > thresh]

    # To drop duplicate pairs (e.g. A-B and B-A), sort and drop duplicates
    strong_pairs[['var1', 'var2']] = strong_pairs[['level_0', 'level_1']].apply(sorted, axis=1, result_type='expand')
    strong_pairs = strong_pairs.drop(columns=['level_0', 'level_1'])
    strong_pairs = strong_pairs.drop_duplicates()

    return strong_pairs.sort_values(by='abs_corr', ascending=False)

strong_corr_pairs = get_strong_predictor_correlations(Xy_train_combined, target='G3', thresh=0.7)
print(strong_corr_pairs)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Train linear regression
lr = LinearRegression()
lr.fit(X_train_combined, y_train)

# Predict and evaluate
y_pred = lr.predict(X_test_combined)

rows , cols = X_test_combined.shape

R2 = r2_score(y_test, y_pred)
Adjusted_R2 = 1 - (1 - R2) * (rows - 1) / (rows - cols - 1)

print("Root Mean Squared Error:", mean_squared_error(y_test, y_pred, squared = False))
print("R² Score:", R2)
print(f"Adjusted R² Score: {Adjusted_R2:.4f}")

In [ ]:
# Combine predictors and target into one DataFrame
df_mlr = X_train_combined.copy()
df_mlr['G3'] = y

# Build formula string: 'Log_medv ~ feature1 + feature2 + ...'
formula = 'G3 ~ ' + ' + '.join(X_train_combined.columns)

# Fit model using formula syntax
model = smf.ols(formula=formula, data=df_mlr).fit()

# Get per-predictor ANOVA report using type 2 (no interaction terms)
anova_report = anova_lm(model, typ=2)

# Display results
print(anova_report)